# Open fragmentation, conservative engine, geometric kernel

The local rule only, $f=0.3$. The non-local $f=0$ case is not analysed here and is not meant to be: without a disruption threshold the grinding rate of a large body is set by the smallest particles in the box, the cascade collapses onto the sink scale, and there is no steady state whose index could be measured.

## Loading

The engine and the whole measurement layer come from `BF_analysis.py` in this folder, so nothing below is defined twice. Every figure is rebuilt from the stored run; no simulation is repeated.

In [ ]:
import os, glob, json, importlib, numpy as np, matplotlib.pyplot as plt

import BF_analysis as AN
#  RELOAD, всегда.  Jupyter кэширует модуль между запусками, и правка BF_analysis.py
#  без перезапуска ядра молча оставляет СТАРЫЙ оценщик -- он не падает, он отвечает
#  иначе.  Это стоило отладочной сессии: срезы приходили пустыми, а всплывало это
#  двадцатью ячейками позже пустым массивом.
importlib.reload(AN)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "figure.figsize": (9, 3.2)})
print("движок:", AN.BF.__name__, "| слой измерения:", AN.__file__)

PATTERN = "runs/open_fragmentation_conservative_geometric_f*.npz"
ONLY_LATEST = True          # <<-- брать ТОЛЬКО самый свежий файл
RUN_FILE    = None          # <<-- или назвать файл явно, тогда ONLY_LATEST не нужен
#  Раньше сюда попадали все файлы под шаблон, включая ручные копии с датой в
#  имени.  Сравнивать прогоны разных движков на одних осях -- отдельная задача,
#  и делать это молча, потому что glob подцепил лишнее, нельзя: половина
#  величин ниже (закон ожидания, честность, поколения) у старых прогонов
#  просто отсутствует, и графики тихо становятся про другое.
_found = sorted(glob.glob(PATTERN), key=os.path.getmtime)
if not _found:
    raise FileNotFoundError("ничего не подходит под %s -- прогони соседнюю run-тетрадь" % PATTERN)
if RUN_FILE is not None:
    _found = [RUN_FILE if os.path.sep in RUN_FILE or "/" in RUN_FILE
              else os.path.join("runs", RUN_FILE)]
elif ONLY_LATEST:
    if len(_found) > 1:
        print("найдено %d файлов, беру последний по времени; остальные пропущены:"
              % len(_found))
        for _f in _found[:-1]:
            print("   пропущен  %s" % os.path.basename(_f))
    _found = _found[-1:]
print("беру:", os.path.basename(_found[0]))
FILES = {os.path.basename(f).replace(".npz", "").split("_")[-1]: f for f in _found}

RUNS, SPEC, GEN, MOM, GRW, WAIT = {}, {}, {}, {}, {}, {}
WEIGHT = "mass"     # <<-- 'mass' (как трейсер) или 'number' (оба осколка поровну)
for tag, fn in FILES.items():
    r = AN.load(fn); RUNS[tag] = r
    SPEC[tag] = AN.spectrum(r)
    print("загружено %-56s stop = %s" % (fn, r["meta"].get("stop_reason")))
    if "gen_counts" in r:
        GEN[tag] = AN.generations(r, weight=WEIGHT)
        MOM[tag] = AN.gen_moments(GEN[tag])
        GRW[tag] = AN.gen_growth(GEN[tag], MOM[tag])
        print("   поколения: снимков %g, пригодных g %d, overflow %.3g"
              % (r["gen_snapshots"], int(MOM[tag]["ok"].sum()), r["gen_overflow"]))
    else:
        print("   в этом прогоне нет гистограммы поколений (движок старше v4)")
    #  [v5] ТЁПЛЫЙ СТАРТ.  Два новых вопроса к каждому файлу, и оба надо задать
    #  ДО того, как читать что-нибудь ещё:
    #    * посеян ли он -- и если да, каким наклоном.  Прогон, посеянный ответом
    #      и отрапортовавший ответ, не доказал ничего;
    #    * какая доля популяции честна.  Всё, что зависит от происхождения --
    #      поколения, изохроны, возрасты -- считается ТОЛЬКО по ней.
    if r["meta"].get("seeded"):
        _sd = r["meta"].get("seed") or {}
        print("   ЗАСЕЯН: alpha_seed = %+.4f на [%.3g, %.3g], N = %s"
              % (_sd.get("alpha", np.nan), _sd.get("m_lo", np.nan),
                 _sd.get("m_hi", np.nan), _sd.get("N", "?")))
    if "honest_num" in r:
        print("   честность: %.3f по числу, %.3f по массе"
              % (float(np.asarray(r["honest_num"])[-1]),
                 float(np.asarray(r["honest_mass"])[-1])))
    if "wait_mean" in r:
        WAIT[tag] = AN.waiting_law(r)
        print("   ожидание: 1/b = %+.5f +- %.5f (стат) +- %.5f (окно) на %.2f декадах"
              % (WAIT[tag]["p"], WAIT[tag]["sigma_p"], WAIT[tag]["p_spread"],
                 WAIT[tag]["decades"]))
    else:
        print("   закона ожидания нет (движок старше v5 или track_waiting=False)")

## What is in the file

Parameters, cost, the analysis stored at save time, and how large the counting error actually is — the empirical scatter beside the Poisson floor it can never go below.

In [ ]:
for tag, r in RUNS.items():
    print(AN.describe(r, tag))
    s = SPEC[tag]
    k = np.isfinite(s["F"]) & (s["F"] > 0)
    print("  %-16s %d snapshots averaged, K_eff median %.0f"
          % ("statistics", s["K"], np.nanmedian(s["K_eff"])))
    print("  %-16s empirical %.4f   Poisson floor %.4f   ratio %.2f"
          % ("median rel. error", np.nanmedian((s["sigma_emp"]/s["F"])[k]),
             np.nanmedian((s["sigma_pois"]/s["F"])[k]),
             np.nanmedian((s["sigma_emp"]/s["F"])[k])
             / max(np.nanmedian((s["sigma_pois"]/s["F"])[k]), 1e-30)))


## Evolution

The population must reach a plateau and the sink counter must climb steadily. $M_{\\rm sys}/m_{\\rm sink}$ is the number that decides whether a steady state is even possible.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.2))
for i, (tag, r) in enumerate(RUNS.items()):
    t = np.asarray(r["t"])
    ax[0].plot(t, r["live"], "-", lw=1.2, color="C%d" % i, label=tag)
    ax[1].plot(t, r["n_out"], "-", lw=1.2, color="C%d" % i, label=tag)
#  t = 0 is the gate: the moment the sink counter said the distribution had reached
#  its functional form, and the moment tagging started.  Everything at negative t is
#  the box filling, and nothing measured is taken from there.
for _a in ax:
    _a.axvline(0.0, ls="--", lw=1, color="0.35")
    _a.annotate("gate", xy=(0, 0), xytext=(2, 2), textcoords="offset points",
                fontsize=6, color="0.35")
ax[0].set_xlabel("t"); ax[0].set_ylabel("live particles"); ax[0].set_ylim(bottom=0)
ax[0].set_title("(a) population -- must plateau")
ax[1].set_xlabel("t"); ax[1].set_ylabel(r"$n_{\rm out}$ absorbed at the sink")
ax[1].set_title("(b) the physical clock of the cascade")
for a in ax: a.legend(fontsize=7)
fig.tight_layout()

for tag, r in RUNS.items():
    msink = r["meta"]["sink_mass"] or np.inf
    print("%-14s live %d -> %d | sink = %d | M_out/M_in = %.3f | M_sys/m_sink = %.1f"
          % (tag, r["live"][0], r["live"][-1], float(r["sink_events"]),
             r["M_out"][-1]/max(r["M_in"][-1], 1), r["M_sys"][-1]/msink))
print("\nM_sys/m_sink below a few means one absorption empties the box: the run is a")
print("relaxation oscillator, not a steady state, and the spectrum is not measurable.")

## Спектр

Полный измеренный спектр, компенсированный как $m^2\,dN/dm$, с ошибками и с линиями
плато и теории, привязанными к одному окну. Поколения лягут на эти же оси в секции ниже.

Маска стационара строится по **счётчику стока** `n_out`, а не по времени ворот: счётчик
нельзя сдвинуть часами. На одном прогоне сохранённое время ворот оказалось вне массива
`t`, каждый снимок провалил проверку, и оценщик молча усреднил **один** снимок из 202.

In [ ]:
#  СПЕКТР.  Полный, компенсированный как m^2 dN/dm, с ошибками, плато и теорией,
#  привязанными к одному окну.  Поколения кладутся на эти же оси в своей секции ниже.
#
#  Маска стационара берётся по СЧЁТЧИКУ СТОКА n_out, а не по времени ворот: счётчик
#  нельзя сдвинуть часами.  На одном прогоне сохранённое время ворот оказалось вне
#  массива t, каждый снимок провалил проверку, и оценщик молча усреднил ОДИН снимок
#  из 202.  Счётчик так сломаться не может.
n = len(RUNS)
fig, axs = plt.subplots(1, n, figsize=(6.0 * n, 3.7), squeeze=False)
for i, (tag, r) in enumerate(RUNS.items()):
    a, s = axs[0][i], SPEC[tag]
    c = np.asarray(r["centers"])
    pl = AN.spectrum_plateau(r, spec=s)
    AN.plot_spectrum(a, r, spec=s, plateau=pl, label=r"измерено $\pm\sigma$")
    at = r["meta"].get("analysis", {}).get("alpha_theory")
    if at and np.isfinite(pl["m_lo"]):
        xs = np.logspace(np.log10(pl["m_lo"]), np.log10(pl["m_hi"]), 30)
        A = AN.anchor_amplitude(c, s["F"], pl["m_lo"], pl["m_hi"], at)
        a.plot(xs, A * xs**at * xs**2, "--", lw=1.4, color="C1", zorder=6,
               label=r"теория $%.2f$" % at)
    w = AN.wls_powerlaw(c, s["F"], s["sigma"], mask=(c >= pl["m_lo"]) & (c <= pl["m_hi"]))
    print("%-10s K = %d снимков | плато alpha = %+.3f +- %.3f на %.2f декадах"
          % (tag, s["K"], pl["alpha"], pl["scatter"], pl["decades"]))
    print("           взвешенный фит %+.4f +- %.4f, chi2/dof = %.2f, n = %d"
          % (w["p"], w["sigma_p"], w["chi2_dof"], w["n"]))
    if w["chi2_dof"] < 0.5:
        print("           chi2/dof < 0.5: снимки коррелированы (K_eff ~ %.0f), так что"
              % np.nanmedian(s["K_eff"]))
        print("           взвешенная ошибка ЗАВЫШЕНА -- бери разброс плато, не её.")
    #  [v5] ЗАСЕВ, пунктиром.  Ради этой одной линии стоит вся возня с честностью:
    #  если измеренное плато лежит НА ней, прогон не измерил alpha, он его повторил.
    _sd = r["meta"].get("seed") or {}
    if _sd and np.isfinite(pl["m_lo"]):
        _as = float(_sd["alpha"])
        xs = np.logspace(np.log10(pl["m_lo"]), np.log10(pl["m_hi"]), 30)
        A = AN.anchor_amplitude(c, s["F"], pl["m_lo"], pl["m_hi"], _as)
        a.plot(xs, A * xs**_as * xs**2, ":", lw=1.6, color="C4", zorder=6,
               label=r"посеяно $%.2f$" % _as)
        print("           посеяно %+.3f -> измерено %+.3f: ушло на %+.3f %s"
              % (_as, pl["alpha"], pl["alpha"] - _as,
                 "(аттрактор сработал)" if abs(pl["alpha"] - _as) > 3 * pl["scatter"]
                 else "-- НЕ УШЛО, засев подсказал ответ"))
    #  Спектр, усреднённый по ВРЕМЕНИ, а не по снимкам.  Интеграл Int N_b dt точный и
    #  достаётся бесплатно -- он же знаменатель закона ожидания, -- поэтому эта кривая
    #  гладкая там, где среднее по снимкам рвано, и взвешена временем, а не тем, как
    #  снимки легли.
    _mt, _Ft = AN.time_avg_spectrum(r)
    if _mt is not None:
        a.plot(_mt, np.where(_Ft > 0, _Ft * _mt**2, np.nan), "-", lw=2.5, alpha=.35,
               color="C2", zorder=2, label="среднее по времени")
    AN.compensated_ylim(a, c, s["F"])
    a.legend(fontsize=6, loc="lower left"); a.set_title(tag)
    a.set_xlabel("m"); a.set_ylabel(r"$m^2\,dN/dm$")
fig.tight_layout()

## Local slope

The diagnostic that decides the fitting window. A genuine inertial range is a plateau in $\\Gamma$; the contaminated ends are where it bends. The two shaded bands must overlap.

In [ ]:
HALF = 3        # half-width of the sliding window, in bins

n = len(RUNS)
fig, axs = plt.subplots(1, n, figsize=(5.2 * n, 2.9), squeeze=False)
for i, (tag, r) in enumerate(RUNS.items()):
    a, s = axs[0][i], SPEC[tag]
    c = np.asarray(r["centers"])
    an = r["meta"].get("analysis", {})
    mm, G = AN.local_slope(c, s["F"], half=HALF)
    a.semilogx(mm, G, "o-", ms=3, lw=.8)
    if an.get("alpha_theory") is not None:
        a.axhline(an["alpha_theory"], ls="--", lw=1, color="k",
                  label=r"theory $%.2f$" % an["alpha_theory"])
    if an.get("guard_lo") and np.isfinite(an["guard_lo"]):
        a.axvspan(an["guard_lo"], an["guard_hi"], alpha=.12, label="guard band")
    pl = AN.spectrum_plateau(r, spec=s)
    if np.isfinite(pl["m_lo"]):
        a.axvspan(pl["m_lo"], pl["m_hi"], alpha=.18, color="C1", label="auto plateau")
    a.set_ylim(-8, 1); a.set_xlabel("m")
    a.set_ylabel(r"$\Gamma = d\log F/d\log m$")
    a.legend(fontsize=6); a.set_title("local slope -- %s" % tag)
fig.tight_layout()


---
## Закон ожидания: $b$ как время

$\langle\Delta t\rangle(m)$ — среднее время между двумя событиями, меняющими массу
частицы, которая сидит при $m$. По однородности ядра
$\nu(m)\sim m^{\lambda+\alpha+1}$, интеграл локален благодаря порогу $f>0$, и
значит $\langle\Delta t\rangle\sim m^{1/b}$.

Три дороги к $b$ печатаются рядом, потому что **болеют они по-разному**:

| дорога | формула | чем портится |
|---|---|---|
| $\tau(g)$ по поколениям | подгонка $\langle\tau\rangle(g)$ | цензура длиной прогона — в v4 дала $1{,}74$ вместо $6$ |
| замыкание | $1/b = 2+\alpha$ | усиление $db/d\alpha=b^{2}=36$ |
| кинетика | $1/b = -(1+\lambda+\alpha)$ | то же усиление, но другая комбинация |
| ожидание | $1/b = d\ln\langle\Delta t\rangle/d\ln m$ | пологий наклон, выбор окна |

Замыкание и кинетика совпадают **только** на неподвижной точке
$\alpha=-(3+\lambda)/2$, так что расхождение между ними — мера того, насколько
прогон до неё не дошёл, а не ошибка ни одной из них.

Две оценки самого $\langle\Delta t\rangle$ — *exposure* (занятость, делённая на
число закончившихся стоянок) и *interval* (среднее по законченным интервалам) —
считаются обе. Первая не смещена цензурой, вторая смещена; если они разошлись,
прогон короче самого долгого времени ожидания.

Панель (b) — локальный наклон — важнее панели (a). Именно на ней видно, что сток
достаёт вверх примерно на декаду: там наклон проваливается к $-2$ и отскакивает к
$+1$, и это обрезание, а не кинетика. Окно фита поэтому несимметрично.


In [ ]:
#  ЗАКОН ОЖИДАНИЯ И ТРИ ДОРОГИ К b.
#  Слева -- сам <dt>(m) с окном фита; серым закрашено то, что окно выбросило.
#  В середине -- локальный наклон, единственная картинка, на которой ВИДНО, где
#  закон есть, а где начинается сток: у стока наклон уходит в -2..-3 и обратно в
#  +1, и это не физика, а обрезание.  Справа -- как уходит посев.
if not WAIT:
    print("ни в одном загруженном прогоне нет закона ожидания (движок старше v5).")
else:
    n = len(WAIT)
    fig, axs = plt.subplots(n, 3, figsize=(14, 3.4 * n), squeeze=False)
    for i, (tag, wl) in enumerate(WAIT.items()):
        r = RUNS[tag]
        lam = float(r["meta"]["lambda"]); p_th = 0.5 * (1 - lam)
        m_inj = float(r["meta"]["injection_mass"]); m_sink = float(r["meta"]["sink_mass"])
        m, dt, nev, msk = wl["m"], wl["dt"], wl["n_events"], wl["mask"]
        ok = np.isfinite(dt) & (nev > 0)

        a = axs[i][0]
        a.loglog(m[ok], dt[ok], "o", ms=3, color="0.5", label="все бины")
        a.loglog(m[msk], dt[msk], "o", ms=4, color="C0", label="в фите")
        xs = np.logspace(np.log10(wl["band"][0]), np.log10(wl["band"][1]), 30)
        a.loglog(xs, wl["A"] * xs ** wl["p"], "-", lw=2, color="C3",
                 label=r"$1/b=%.4f\pm%.4f$" % (wl["p"], wl["sigma_p"]))
        A0 = AN.anchor_amplitude(m, dt, wl["band"][0], wl["band"][1], p_th)
        a.loglog(xs, A0 * xs ** p_th, "--", lw=1.3, color="C1",
                 label=r"теория $1/b=%.4f$" % p_th)
        a.axvspan(min(m_sink, m_inj), wl["band"][0], color="0.85", zorder=0)
        a.axvspan(wl["band"][1], max(m_sink, m_inj), color="0.85", zorder=0)
        a.set_xlabel("m"); a.set_ylabel(r"$\langle\Delta t\rangle$")
        a.legend(fontsize=6); a.set_title("%s: (a) закон ожидания" % tag, fontsize=9)

        a = axs[i][1]
        L = np.log(m); LY = np.log(np.where(dt > 0, dt, np.nan))
        sl = np.full_like(L, np.nan)
        sl[1:-1] = (LY[2:] - LY[:-2]) / (L[2:] - L[:-2])
        a.semilogx(m, sl, "o-", ms=3, lw=.8, color="C0")
        a.axhline(p_th, ls="--", color="C1", lw=1.3, label=r"теория $%.4f$" % p_th)
        a.axhline(wl["p"], ls="-", color="C3", lw=1.3, label="фит %.4f" % wl["p"])
        a.axvspan(min(m_sink, m_inj), wl["band"][0], color="0.85", zorder=0)
        a.axvspan(wl["band"][1], max(m_sink, m_inj), color="0.85", zorder=0)
        a.set_ylim(-1.0, 1.0); a.set_xlabel("m")
        a.set_ylabel(r"$d\ln\langle\Delta t\rangle/d\ln m$")
        a.legend(fontsize=6); a.set_title("(b) локальный наклон = 1/b", fontsize=9)

        a = axs[i][2]
        H = AN.honesty(r)
        if H is not None:
            a.plot(H["n_out"], H["num"], "o-", ms=3, lw=1, label="по числу")
            a.plot(H["n_out"], H["mass"], "s-", ms=3, lw=1, label="по массе")
            a.axhline(1.0, ls=":", color="0.4"); a.set_ylim(-0.02, 1.05)
            a.set_xlabel(r"$n_{\rm out}$"); a.set_ylabel("честная доля")
            a.legend(fontsize=6)
        a.set_title("(c) как уходит посев", fontsize=9)
    fig.tight_layout()

    # ---- таблица: три дороги к b, по одному блоку на прогон ----------------
    for tag, wl in WAIT.items():
        r = RUNS[tag]
        pl = AN.spectrum_plateau(r, spec=SPEC[tag])
        print(AN.waiting_report(r, wl, alpha=pl["alpha"], sigma_alpha=pl["scatter"]))
        lam = float(r["meta"]["lambda"]); al, sal = pl["alpha"], pl["scatter"]
        rows = [("теория",                     0.5 * (1 - lam),   np.nan),
                ("<dt>(m), измерено",          wl["p"],           wl["sigma_p"]),
                ("замыкание   2+alpha",        2 + al,            sal),
                ("кинетика  -(1+lam+alpha)",  -(1 + lam + al),    sal)]
        if tag in GRW and np.isfinite(GRW[tag]["b"]):
            rows.append(("tau(g) по поколениям", 1.0 / GRW[tag]["b"], np.nan))
        print("\n%-26s %12s %10s %10s %10s" % ("дорога", "1/b", "sigma", "b", "sigma_b"))
        for lab, p, sp in rows:
            bb = 1.0 / p if p else np.nan
            print("%-26s %+12.5f %10.5f %10.3f %10.3f" % (lab, p, sp, bb, bb * bb * sp))
        print("\nsigma_b = b^2 sigma_(1/b) -- читать строку 1/b, а не строку b.")
        print("Три дороги болеют по-разному: tau(g) цензурируется длиной прогона,")
        print("замыкание усилено в b^2 раз, <dt> ни то ни другое -- но у него своё окно,")
        print("и systematics окна (%.5f) здесь %s статистики (%.5f)."
              % (wl["p_spread"],
                 "БОЛЬШЕ" if wl["p_spread"] > wl["sigma_p"] else "меньше",
                 wl["sigma_p"]))


---
## Поколения

Номер разлома несёт **каждая** частица, один int32 — та же величина, ради которой в
v2/v3 строились трейсеры, но снятая со всей популяции. Три ограничения трейсеров
исчезают, а не решаются: число независимых деревьев (его нельзя было поднять выше
$n_{\rm out}/R$), полка у $m_{\rm inj}$, где сидело 92 % времени пребывания, и
цензурирование стоком.

**Взвешивание — единственное, что надо не перепутать.** `gen_counts` считает оба
осколка: это по **числу**, и шаг равномерного дробления даёт $\langle\ln\xi\rangle=-1$,
${\rm Var}=1$. `gen_mass` — по массе, ровно как трейсер, следовавший за куском с
вероятностью $\xi$: $-1/2$ и $1/4$. По массе взято по умолчанию ещё и потому, что пакет
едет вдвое медленнее и держится выше стока вдвое дольше по поколениям.

Панель (b) второй картинки — **где сток начинает есть**. Дробление удваивает число
частиц каждое поколение, поэтому число достигших $g$ обязано идти как $2^g$, пока сток
не начнёт забирать осколки раньше, чем они успеют разделиться. Там, где отношение падает
ниже двух, до $g$ доходят только **быстрые** — смещение выжившего, которое сжимает
$\tau(g)$ и тянет $b$ вниз. Фит идёт только по плоскому участку.

$b$ печатается рядом с $b$ из замыкания на $\alpha$ **с прокинутой ошибкой**, потому что
$db/d\alpha=b^2$ — множитель тридцать шесть при $b=6$.

In [ ]:
#  ПОКОЛЕНИЯ: распределения по массе, моменты, часы и коллапс.
GEN_SHOW = 12
for tag in GEN:
    r, s, g, m, G = RUNS[tag], SPEC[tag], GEN[tag], MOM[tag], GRW[tag]
    W_ = float(r["meta"]["frag_split_width"]); MU_TH, VAR_TH = AN.step_stats(W_, WEIGHT)
    c = g["centers"]; pl = AN.spectrum_plateau(r, spec=s); minj = g["m_inj"]
    print("=" * 92)
    print("%s  взвешивание %s: <ln xi> = %.4f, Var = %.4f (точно)" % (tag, WEIGHT, MU_TH, VAR_TH))
    print("%4s %11s %9s %9s %10s %10s %9s" % ("g", "вес", "<x>", "теор", "Var", "теор", "reach"))
    for gi in g["g"][:GEN_SHOW + 6]:
        if not np.isfinite(m["mu"][gi]):
            continue
        print("%4d %11.4g %9.3f %9.3f %10.3f %10.3f %9.3g%s"
              % (gi, m["n"][gi], m["mu"][gi], MU_TH * gi, m["var"][gi], VAR_TH * gi,
                 m["reach_n"][gi], "" if m["ok"][gi] else "   <- пакет у стока"))

    gs = g["g"][m["ok"]][:GEN_SHOW]
    cm = plt.cm.viridis(np.linspace(0.04, 0.94, max(len(gs), 1)))
    tot = g["H"].sum(axis=0)
    k0 = (c >= pl["m_lo"]) & (c <= pl["m_hi"]) & (tot > 0) & (s["F"] > 0)
    A = (np.median((s["F"][k0] * c[k0]**2) / (tot[k0] / g["widths"][k0] * c[k0]))
         if k0.sum() >= 3 else np.nan)

    fig, ax = plt.subplots(1, 3, figsize=(14.5, 3.8))
    for col, gi in zip(cm, gs):
        y = A * g["H"][gi] / g["widths"] * c
        ax[0].loglog(c, np.where(y > 0, y, np.nan), "-", lw=1.2, color=col, label="g = %d" % gi)
    yt = A * tot / g["widths"] * c
    ax[0].loglog(c, np.where(yt > 0, yt, np.nan), "-", lw=3.5, alpha=.35, color="0.4",
                 label="сумма по всем g")
    ax[0].loglog(c, np.where(s["F"] > 0, s["F"] * c**2, np.nan), "o", ms=2.5, color="k",
                 label="стационарный спектр")
    AN.compensated_ylim(ax[0], c, s["F"])
    ax[0].set_xlabel("m"); ax[0].set_ylabel(r"$m^2\,dN/dm$")
    ax[0].legend(fontsize=5.5, ncol=2, loc="lower left")
    ax[0].set_title("(a) поколения на спектре -- %s" % tag, fontsize=9)

    dx = np.log(c[1] / c[0])
    for col, gi in zip(cm, gs):
        h = g["H"][gi]
        if h.sum() <= 0:
            continue
        ax[1].semilogy(np.log(c / minj), h / h.sum(), "-", lw=1.2, color=col)
        xx = np.linspace(MU_TH*gi - 4*np.sqrt(VAR_TH*gi), MU_TH*gi + 4*np.sqrt(VAR_TH*gi), 200)
        ax[1].semilogy(xx, dx * np.exp(-(xx - MU_TH*gi)**2 / (2*VAR_TH*gi))
                       / np.sqrt(2*np.pi*VAR_TH*gi), "--", lw=.9, color=col, alpha=.75)
    ax[1].axvline(m["x_sink"], ls=":", color="0.3", lw=1.2)
    ax[1].set_ylim(1e-5, 1); ax[1].set_xlabel(r"$x=\ln(m/m_{\rm inj})$")
    ax[1].set_ylabel("доля поколения")
    ax[1].set_title(r"(b) пунктир $N(\mu g,\sigma^2 g)$, без подгонки", fontsize=9)

    kk = m["ok"]; gg = g["g"][kk].astype(float)
    ax[2].plot(gg, m["mu"][kk], "o", ms=4, color="C0", label=r"$\langle x\rangle$")
    ax[2].plot(gg, m["var"][kk], "s", ms=4, color="C1", label=r"${\rm Var}(x)$")
    ax[2].plot(gg, MU_TH*gg, "--", lw=1.2, color="C0", label=r"$%.3f\,g$" % MU_TH)
    ax[2].plot(gg, VAR_TH*gg, "--", lw=1.2, color="C1", label=r"$%.3f\,g$" % VAR_TH)
    ax[2].set_xlabel("g"); ax[2].legend(fontsize=6)
    ax[2].set_title("(c) обе линейны по g", fontsize=9)
    fig.tight_layout()

    fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
    if np.isfinite(G["b"]):
        ax[0].plot(G["g"], G["tau"], "o", ms=5, label=r"$\langle\tau\rangle(g)$")
        _g = np.linspace(0, G["g"].max(), 200)
        ax[0].plot(_g, G["T"]*(1-np.exp(-_g/G["scale"])), "-", lw=1.8, color="C3",
                   label="фит b = %.2f (resid %.3f)" % (G["b"], G["resid"]))
        bt = r["meta"].get("analysis", {}).get("b_theory")
        if bt:
            ax[0].plot(_g, G["T"]*(1-np.exp(-_g/(bt/abs(MU_TH)))), "--", lw=1.3,
                       color="C1", label="теория b = %.2f" % bt)
        ax[0].legend(fontsize=6)
    ax[0].set_xlabel("g"); ax[0].set_ylabel(r"$\langle\tau\rangle$")
    ax[0].set_title("(a) время достижения поколения", fontsize=9)

    rn = m["reach_n"]
    ax[1].semilogy(g["g"][1:], np.where(rn[1:] > 0, rn[1:], np.nan), "o-", ms=3, lw=1)
    if rn[1] > 0:
        ax[1].semilogy(g["g"][1:], rn[1] * 2.0**(g["g"][1:] - 1), "--", lw=1.2, color="C3",
                       label=r"$2^g$ -- дробление удваивает")
        ax[1].legend(fontsize=6)
    ax[1].set_xlabel("g"); ax[1].set_ylabel("достигло g")
    ax[1].set_title("(b) где сток начинает есть", fontsize=9)

    for col, gi in zip(cm, gs):
        h = g["H"][gi]
        if h.sum() <= 0 or not np.isfinite(m["var"][gi]) or m["var"][gi] <= 0:
            continue
        z = (np.log(c / minj) - m["mu"][gi]) / np.sqrt(m["var"][gi])
        ax[2].semilogy(z, h / h.sum() / (dx / np.sqrt(m["var"][gi])), "-", lw=1.2, color=col)
    zz = np.linspace(-4.5, 4.5, 300)
    ax[2].semilogy(zz, np.exp(-zz**2/2)/np.sqrt(2*np.pi), "k--", lw=1.4, label="N(0,1)")
    ax[2].set_xlim(-4.5, 4.5); ax[2].set_ylim(1e-4, 1); ax[2].legend(fontsize=6)
    ax[2].set_xlabel(r"$z=(x-\mu g)/(\sigma\sqrt{g})$")
    ax[2].set_title("(c) коллапс", fontsize=9)
    fig.tight_layout()

    bt = r["meta"].get("analysis", {}).get("b_theory")
    if np.isfinite(G["b"]):
        print("  b = %.2f из tau(g) на g = %d..%d (resid %.3f против прямой %.3f)"
              % (G["b"], int(G["g"].min()), int(G["g"].max()), G["resid"], G["resid_linear"]))
    beta = -(1.0 + pl["alpha"]); bc = 1.0/(1.0-beta) if beta < 1 else np.inf
    print("  b = %.1f +- %.1f из замыкания на alpha = %+.3f +- %.3f   (db/dalpha = b^2)"
          % (bc, bc**2 * pl["scatter"], pl["alpha"], pl["scatter"]))
    if bt:
        need = bt / abs(MU_TH)
        print("  масштаб tau(g) = %.0f поколений, пригодных %d -- %s"
              % (need, int(m["ok_flux"].sum()),
                 "хватает" if m["ok_flux"].sum() > need else "МАЛО, b не определится"))

## Mass budget

Where the mass sits and whether it balances. Closed: $M_{\\rm sys}$ constant to machine precision. Open: $M_{\\rm out}/M_{\\rm in}\\to1$ is the definition of the steady state.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.2))
for i, (tag, r) in enumerate(RUNS.items()):
    t = np.asarray(r["t"])
    ax[0].plot(t, r["M_sys"], "-", lw=1.2, color="C%d" % i, label="%s: in the box" % tag)
    ax[0].plot(t, r["M_in"], "--", lw=1, color="C%d" % i, label="%s: injected" % tag)
    ax[0].plot(t, r["M_out"], ":", lw=1.4, color="C%d" % i, label="%s: absorbed" % tag)
    denom = np.where(np.asarray(r["M_in"]) > 0, r["M_in"], np.nan)
    ax[1].plot(t, np.asarray(r["M_out"]) / denom, "-", lw=1.2, color="C%d" % i, label=tag)
ax[0].set_xlabel("t"); ax[0].set_ylabel("mass"); ax[0].legend(fontsize=6)
ax[0].set_title("(a) where the mass is")
ax[1].axhline(1.0, ls="--", lw=1, color="k")
ax[1].set_xlabel("t"); ax[1].set_ylabel(r"$M_{\rm out}/M_{\rm in}$"); ax[1].legend(fontsize=7)
ax[1].set_title("(b) steady state means this reaches 1")
fig.tight_layout()

for tag, r in RUNS.items():
    print("%-14s M_sys %.6g -> %.6g | M_in %.4g | M_out %.4g | drift %+.2e"
          % (tag, r["M_sys"][0], r["M_sys"][-1], r["M_in"][-1], r["M_out"][-1],
             float(r["mass_drift"])))


## Evolution of the spectrum

Selected snapshots, compensated.

In [ ]:
N_SNAP = 8        # <<-- how many snapshots to draw per run

n = len(RUNS)
fig, axs = plt.subplots(1, n, figsize=(5.4 * n, 3.4), squeeze=False)
for i, (tag, r) in enumerate(RUNS.items()):
    a = axs[0][i]
    c = np.asarray(r["centers"]); D = np.asarray(r["dndm"]); t = np.asarray(r["t"])
    keep = [k for k in range(D.shape[0]) if np.any(D[k] > 0)]
    sel = np.unique(np.linspace(0, len(keep) - 1, min(N_SNAP, len(keep))).astype(int))
    cm = plt.cm.plasma(np.linspace(0, .88, len(sel)))
    for col, sidx in zip(cm, sel):
        kk = keep[sidx]
        a.loglog(c, np.where(D[kk] > 0, D[kk] * c**2, np.nan), lw=1.1, color=col,
                 label="t = %.3g" % t[kk])
    a.set_xlabel("m"); a.set_ylabel(r"$m^2\,dN/dm$")
    a.legend(fontsize=5.5, ncol=2); a.set_title("evolution -- %s" % tag)
fig.tight_layout()


---
## Итог: спектр и поколения в одних осях (стиль PRL)

Ничего, кроме этого:

- спектр — $dN/dm$, умножить на $m^{2}$;
- поколение $g$ — $(dN/dm)_g$, умножить на $m^{2}$.

Никаких нормировок. Поколения — разбиение популяции на непересекающиеся куски,
поэтому каждое лежит **ниже** спектра, а их сумма его восстанавливает.

**С одной оговоркой, и она физическая.** Поколение считается только у частиц,
вошедших после $t=0$; посеянные несут `gen = -1`, и метка наследуется. На этом
прогоне посева ещё 99 % по числу, поэтому серая линия — сумма **всех**
поколений — идёт заметно ниже чёрных кружков. Разность чёрных и серой и есть
посев. Когда он уйдёт, серая сядет на спектр. Численный контроль печатается под
рисунком: сумма поколений / спектр $= 0{,}0068$, ровно честная доля по числу.

Заметьте, как серая линия подходит к спектру справа, у инжекции: там честных
почти половина — частицы только что вошли и не успели раздробиться. У стока
честных полпроцента. Это и есть та причина, по которой честных $0{,}85\,\%$ по
числу, но $26\,\%$ по массе.

Ось массы обычная, растёт вправо: сток слева, инжекция справа. При
$\alpha=-1.87$ величина $m^{2}dN/dm$ идёт как $m^{+0.13}$, то есть слабо
поднимается — измерено $+0.117$. Показатель в подписи относится к $dN/dm$, а не к
нарисованной кривой; их наклоны различаются на двойку.

Цвет — $\tau(g)$, возраст в **момент достижения** $g$ (`gen_tau`), а не средний
возраст сидящих на $g$ сейчас (`gen_tau_live` насыщается на времени пребывания
коробки и шкала схлопывается). Штриховые вертикали — инжекция и сток.

Замена прежнему рисунку по изохронам: изохрона — срез по возрасту, в неё попадает
лишь доля популяции из данного возрастного бина, а поколение несёт **каждая**
частица одним `int32`. На этом прогоне в изохронах статистика на бин — десятки,
в поколениях — сотни тысяч.

Какие поколения рисовать — массив `GEN_PICK` в первых строках ячейки.

In [ ]:
# ======================================================================
#  ИТОГ, стиль PRL:  спектр и поколения, окрашенные возрастом поколения
# ======================================================================
#  Ничего, кроме этого, тут не делается:
#     спектр       -> dN/dm,           умножить на m^2
#     поколение g  -> (dN/dm)_g,       умножить на m^2
#  Никаких нормировок.  Поколения -- это просто разбиение популяции на
#  непересекающиеся куски, поэтому каждое лежит НИЖЕ спектра, а их сумма
#  спектр восстанавливает.  Ровно с одной оговоркой, и она физическая:
#
#  СУММА ПОКОЛЕНИЙ НЕ ДОТЯГИВАЕТ ДО СПЕКТРА НА ПОСЕВ.  Поколение считается
#  только у частиц, вошедших после t = 0; посеянные несут gen = -1, и метка
#  наследуется (движок, заметка [13]).  На этом прогоне посева ещё 99% по
#  числу, поэтому серая линия -- сумма всех поколений -- идёт заметно ниже
#  чёрных кружков.  Разность чёрных и серой и есть посев.  Когда он уйдёт,
#  серая линия сядет на спектр.
#
#  Ось цвета -- tau(g), возраст в МОМЕНТ достижения поколения g (`gen_tau`),
#  а не средний возраст сидящих на g сейчас (`gen_tau_live` насыщается на
#  времени пребывания коробки, и шкала схлопывается).
#
#  Спектр нарисован без моделей: точки с ошибками и ничего больше.  Показатель
#  стоит подписью, как число, а не как проведённая через данные кривая.

# ---- ЧТО РИСОВАТЬ.  Единственное, что тут стоит трогать. ---------------
GEN_PICK    = (4, 8,14, 18,24)   # <<-- КАКИЕ ПОКОЛЕНИЯ.  Меняйте здесь.
GEN_REBIN   = 3        # слить по стольку исходных бинов (0.1 dex -> 0.3 dex)
MIN_CNT     = 30.0     # порог по СЫРЫМ отсчётам (частица-снимок) на бин
AGE_UNIT    = 1e-6     # единица подписи цветовой шкалы
SHOW_HONEST = True     # серая линия: сумма ВСЕХ поколений
TAG = None             # None = единственный загруженный прогон

from matplotlib.colors import LogNorm

tag = TAG if TAG is not None else list(RUNS)[-1]
r, s = RUNS[tag], SPEC[tag]
if "gen_counts" not in r:
    raise KeyError("в прогоне %r нет гистограммы поколений (движок старше v4). "
                   "Загрузилось не то: проверь RUN_FILE / ONLY_LATEST выше." % tag)

m   = np.asarray(r["centers"], float)
ed  = np.asarray(r["edges"], float)
H   = np.asarray(r["gen_counts"], float)   # (g, bin): отсчёты, только честные частицы
nsn = max(float(r["gen_snapshots"]), 1.0)  # сколько снимков просуммировано
tau_g  = np.asarray(r["gen_tau"], float)
m_inj  = float(r["meta"]["injection_mass"])
m_sink = float(r["meta"]["sink_mass"])


def _rebin(F, edges, k):
    """Слить по k бинов, сохранив ПЛОТНОСТЬ: суммируем счёт, делим на новую ширину.
    Возвращает (центры, плотность, СЫРЫЕ отсчёты) -- отсчёты нужны для порога."""
    n = (edges.size - 1) // k * k
    w = np.diff(edges)[:n]
    cnt = (np.asarray(F)[:n] * w).reshape(-1, k).sum(1)
    e2 = np.concatenate([edges[:n:k], [edges[n]]])
    return np.sqrt(e2[:-1] * e2[1:]), cnt / np.diff(e2), cnt


pick = np.array([q for q in GEN_PICK
                 if 0 <= q < H.shape[0] and H[q].sum() > 0 and np.isfinite(tau_g[q])])
if pick.size == 0:
    raise RuntimeError("ни одно из GEN_PICK не имеет статистики или tau(g): %r" % (GEN_PICK,))
skipped = [q for q in GEN_PICK if q not in pick]
if skipped:
    print("пропущены (нет статистики или tau = nan; g = 0 не 'достигается'):", skipped)
age = tau_g[pick] / AGE_UNIT

prl_rc = {
    "figure.dpi": 160, "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "legend.fontsize": 6.5, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "axes.linewidth": 0.8, "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True, "axes.grid": False,
}

with plt.rc_context(prl_rc):
    fig = plt.figure(figsize=(3.35, 2.45))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 0.045], wspace=0.07)
    ax = fig.add_subplot(gs[0, 0]); cax = fig.add_subplot(gs[0, 1])

    # ---- спектр: m^2 dN/dm, точки с ошибками, без моделей ---------------
    ok_s = np.isfinite(s["F"]) & (s["F"] > 0) & np.isfinite(m) & (m > 0)
    ax.errorbar(m[ok_s], (s["F"] * m**2)[ok_s], yerr=(s["sigma"] * m**2)[ok_s],
                fmt="o", ms=2.0, mfc="white", mec="0.25", mew=0.5,
                ecolor="0.75", elinewidth=0.45, capsize=1.0, color="0.25", zorder=4)
    y_all = [(s["F"] * m**2)[ok_s]]

    # ---- сумма всех поколений = честная подвыборка ----------------------
    if SHOW_HONEST:
        mh, Fh_raw, ch = _rebin(H.sum(0) / np.diff(ed), ed, GEN_REBIN)
        Fh = Fh_raw / nsn
        okh = np.isfinite(Fh) & (Fh > 0) & (ch >= MIN_CNT)
        ax.loglog(mh[okh], (Fh * mh**2)[okh], "-", lw=1.1, color="0.60", zorder=2)
        y_all.append((Fh * mh**2)[okh])

    # ---- отдельные поколения, цвет по tau(g) ----------------------------
    norm = LogNorm(vmin=age.min(), vmax=max(age.max(), age.min() * 1.0001))
    cmap = plt.cm.viridis
    for gg, tk in zip(pick, age):
        mr, Fr_raw, cnt = _rebin(H[gg] / np.diff(ed), ed, GEN_REBIN)
        Fr = Fr_raw / nsn
        ok = np.isfinite(Fr) & (Fr > 0) & (cnt >= MIN_CNT)
        if not ok.any():
            continue
        ax.loglog(mr[ok], (Fr * mr**2)[ok], "-", ms=2.6, markeredgewidth=0.0,
                  color=cmap(norm(tk)), alpha=0.92, zorder=3)
        y_all.append((Fr * mr**2)[ok])

    # ---- инжекция и сток ------------------------------------------------
    ax.axvline(m_inj, color="0.1", lw=0.8, ls="--", zorder=1)
    ax.text(m_inj / 1.35, 0.965, "Injection", transform=ax.get_xaxis_transform(),
            ha="right", va="top", fontsize=7, color="0.1")
    ax.axvline(m_sink, color="0.1", lw=0.8, ls="--", zorder=1)
    ax.text(m_sink * 1.35, 0.965, "Sink", transform=ax.get_xaxis_transform(),
            ha="left", va="top", fontsize=7, color="0.1")

    #  Подписан показатель СПЕКТРА.  Нарисовано m^2 dN/dm, её собственный
    #  наклон равен 2 + alpha = +0.13, то есть кривая слабо РАСТЁТ с массой.
    pl = AN.spectrum_plateau(r, spec=s)
    ax.text(0.30, 0.30, r"$dN/dm \propto m^{%.2f}$" % pl["alpha"],
            transform=ax.transAxes, ha="center", va="center", fontsize=7.5, color="0.1")

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cb = fig.colorbar(sm, cax=cax)
    #  Тики -- сами tau(g) выбранных поколений, соседние прореживаются: на
    #  больших g возрасты сходятся и подписи иначе садятся друг на друга.
    la = np.log10(age); keep = [0]
    for i in range(1, la.size):
        if (la[i] - la[keep[-1]]) > 0.055 * (la.max() - la.min() + 1e-12):
            keep.append(i)
    if keep[-1] != la.size - 1:
        keep[-1] = la.size - 1
    cb.set_ticks(age[keep]); cb.set_ticklabels(["%.2f" % a for a in age[keep]])
    cb.minorticks_off()
    cb.set_label(r"$\tau(g)$  ($10^{%d}$)" % round(np.log10(AGE_UNIT)),
                 labelpad=2, fontsize=7)
    cb.ax.tick_params(direction="in", length=2.5, width=0.7, labelsize=6.5)

    yp = np.concatenate([np.asarray(v)[np.isfinite(v) & (np.asarray(v) > 0)] for v in y_all])
    ax.set_xlim(m_sink / 2.5, m_inj * 2.5)
    ax.set_ylim(10.0 ** 4,
                10.0 ** np.ceil(np.log10(yp.max()) + 0.15))
    ax.set_xlabel(r"$m$"); ax.set_ylabel(r"$m^{2}\,dN/dm$")
    fig.subplots_adjust(left=0.175, right=0.865, bottom=0.175, top=0.98, wspace=0.07)

# ---- контроль: складываются ли поколения обратно в спектр --------------
tot_gen = float((H.sum(0) / nsn).sum())              # честных частиц на снимок
tot_spec = float((s["F"] * np.diff(ed)).sum())       # всего частиц в спектре
print("прогон : %s" % tag)
print("g      : %s" % list(map(int, pick)))
print("tau(g) : %s   (единица %g)" % (["%.3f" % a for a in age], AGE_UNIT))
print("отсчётов в поколении: %s" % ["%.3g" % H[q].sum() for q in pick])
#  ЗАМЫКАНИЕ, и сравнивать надо ОДНО И ТО ЖЕ усреднение.  gen_counts копится
#  на каждом снимке, где открыты ворота, а AN.spectrum усредняет по маске
#  стационара -- это другой набор снимков.  Поэтому здесь спектр берётся как
#  среднее всех строк dndm, ровно тех, на которых копились поколения.
_w = np.diff(ed)
tot_gen = float((H.sum(0) / nsn).sum())
tot_spec = float((np.asarray(r["dndm"]).mean(axis=0) * _w).sum())
_ratio = tot_gen / tot_spec
_hn = float(np.asarray(r["honest_num"])[-1])
print("\nзамыкание суммы:")
print("  сумма всех поколений       %.5g частиц на снимок" % tot_gen)
print("  весь спектр                %.5g" % tot_spec)
print("  отношение                  %.5f" % _ratio)
if _ratio > 0.99:
    print("  ЕДИНИЦА: поколения замыкаются в спектр точно.  Так и должно быть на")
    print("  холодном старте и на перезапуске -- история честна у всех.")
else:
    print("  МЕНЬШЕ ЕДИНИЦЫ на величину ПОСЕВА: частицы с gen = -1, вошедшие до")
    print("  t = 0, и их потомки.  Совпадает с honest_num = %.5f." % _hn)
    print("  по массе честных %.1f%% -- посев сидит внизу, где частиц много,"
          % (100 * float(np.asarray(r["honest_mass"])[-1])))
    print("  а массы мало.")
if r["meta"].get("restarted_from"):
    print("  прогон вырос из %s" % r["meta"]["restarted_from"])
    print("  коробка прожила %.4g до этого прогона -- поэтому tau(g) не обрезана её длиной"
          % r["meta"].get("t_origin", 0.0))
print("бин %.2f dex (слито по %d), порог %g отсчётов на бин"
      % (np.log10(ed[GEN_REBIN] / ed[0]), GEN_REBIN, MIN_CNT))
